# 빌드업 2편 — 그림체 주입을 "진짜 검색"으로 바꿔봤다

1편 뼈대에서 **그림체 주입 노드만 진짜로** 바꿔본 기록이다.
1편에서는 그림체 문구 사전으로 흉내만 냈는데, 이번엔 코퍼스(daypack_v2)에서
**CLIP 임베딩으로 참조 이미지를 실제로 찾아온다.** Visual RAG 의 R(검색)이 처음으로 진짜가 됐다.

```
1편  대본 작성 -> 그림체 주입(흉내) -> 이미지 생성(자리표시) -> 채점(흉내) + 루프
2편  대본 작성 -> 그림체 주입(★진짜 검색) -> 이미지 생성(자리표시) -> 검색 품질 확인
다음  생성을 진짜로(비용 합의 후) -> 채점을 생성물에 -> 루프 부활
```

- 1편의 재시도 루프는 이번 편에서 잠시 뺐다. 루프의 손잡이(프롬프트 보강)가 생성이 진짜일 때
  의미가 생겨서, 생성을 붙일 때 다시 살리려고 한다
- 준비물: 드라이브 공유 폴더 `DLthon_그림체RAG` 바로가기 (시작 노트북 때 해뒀으면 그대로 된다).
  없어도 목 모드로 끝까지 돈다
- **런타임 유형을 GPU 로** 해두면 임베딩이 빠르다 (런타임 -> 런타임 유형 변경 -> T4).
  CPU 여도 표본만 써서 몇 분이면 된다


## 0. 설치 · 코퍼스 · 키

설치는 1편과 같고, CLIP 은 코랩에 기본으로 있는 transformers 를 쓴다.


In [ ]:
# 랭그래프 + LLM 호출용 설치. 자리표시 이미지의 한글 폰트도 1편처럼 챙긴다.
# transformers 는 5.x 가 CLIP 반환형을 바꿔서 우리 코드가 깨진다 -> 4.x 로 고정 (검증된 버전대)
!pip install -q langgraph langchain-openai "transformers<5"
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1


In [ ]:
import os, glob

# 팀 코퍼스(daypack_v2)를 찾는다. 시작 노트북과 같은 방식이다.
# 어디서도 못 찾으면 MOCK_CORPUS=True 로 두고 1편처럼 문구 사전으로 물러난다 -> 노트북은 끝까지 돈다.
PACK = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # 폴더 이름이 아니라 "daypack_v2.zip 이 들어 있는 DLthon* 폴더"를 찾는다
    # (폴더를 옮기거나 이름이 바뀌어도 내 드라이브 3단계 안이면 잡힌다)
    cands = []
    for root in ['/content/drive/MyDrive/', '/content/drive/.shortcut-targets-by-id/*/']:
        for depth in ['', '*/', '*/*/', '*/*/*/']:
            cands += glob.glob(root + depth + 'DLthon*')
    src_dir = next((c for c in cands if os.path.exists(os.path.join(c, 'daypack_v2.zip'))), None)
    if src_dir:
        import zipfile
        print("코퍼스 폴더:", src_dir)
        with zipfile.ZipFile(os.path.join(src_dir, 'daypack_v2.zip')) as f:
            f.extractall('/content')            # 드라이브에서 바로 읽으면 느려서 코랩 로컬에 푼다
        PACK = '/content/daypack_v2'
except ModuleNotFoundError:
    # 코랩이 아닌 환경(참고용): 옆에 코퍼스 폴더가 있으면 그걸 쓴다
    for c in ['daypack_v2', '../daypack_v2']:
        if os.path.exists(os.path.join(c, 'meta.csv')):
            PACK = c
            break

MOCK_CORPUS = PACK is None
if MOCK_CORPUS:
    print("코퍼스를 못 찾았습니다 -> 목 모드로 진행합니다 (끝까지 돌아는 갑니다).")
    print("진짜 검색을 보려면: 드라이브에서 'DLthon_그림체RAG' 우클릭 -> 내 드라이브에 바로가기 추가 -> 이 셀 재실행")
else:
    n = len(glob.glob(PACK + '/images/*/*.jpg'))
    print(f"코퍼스 연결: {PACK} (그림 {n}장)")


In [ ]:
# 1편과 같은 키 셀. 없으면 대본 작성이 목으로 돈다.
OPENAI_API_KEY = None
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_KEY')
except Exception:
    pass
MOCK = OPENAI_API_KEY is None
print("대본 작성:", "목(mock)" if MOCK else "실제 LLM", "/ 그림체 검색:", "목(mock)" if MOCK_CORPUS else "진짜 검색")


## 1. 검색이 실제로 하는 일

코퍼스에는 그림이 20가지 그림체 클래스로 나뉘어 들어 있다 (`meta.csv` 가 그 명단).
"이 그림체로 그려줘"에 쓸 **참조 이미지**를 골라야 하는데, 클래스 안에도 그림이 100장 넘게 있어서
아무거나 집으면 그 클래스답지 않은 그림이 걸릴 수 있다.

그래서 이렇게 골랐다:

1. 클래스 그림들을 전부 **CLIP 임베딩**(그림 -> 숫자 벡터)으로 바꾼다
2. 벡터들의 **평균(= 그 그림체의 중심)**을 구한다
3. **중심에 가장 가까운 그림 k 장**을 참조로 쓴다 -> 그 클래스에서 제일 "그 그림체다운" 그림들

시작 노트북에서 돌린 채점기(score.py)가 쓰는 것과 같은 임베딩이다. 채점기는 전체를 재는 거라
전 장을 임베딩하지만, 여기선 참조를 고르는 게 목적이라 **클래스당 표본 30장**이면 충분했다.

(참고로 CLIP 은 텍스트 탑이 영어라서 "수묵화 느낌" 같은 한국어 문장 검색은 바로 안 된다.
장면 내용으로 참조를 고르는 건 나중 숙제로 적어둔다.)


In [ ]:
import csv, collections, random

if not MOCK_CORPUS:
    # 코퍼스 명단을 읽는다. style = 그림체 클래스, group = 같은 원본에서 나온 그림 묶음(누수 차단용)
    rows = list(csv.DictReader(open(f'{PACK}/meta.csv', encoding='utf-8')))
    by_style = collections.defaultdict(list)
    for r in rows:
        by_style[r['style']].append(r['file'])
    print(f"클래스 {len(by_style)}개 / 총 {len(rows)}장")
    for k in sorted(by_style):
        print(f"  {k:28s} {len(by_style[k])}장")
else:
    by_style = {}
    print("(목 모드 - 코퍼스 명단 없음)")


In [ ]:
if not MOCK_CORPUS:
    # 채점기(kit)와 같은 CLIP 인코더. 학습 0회, 그대로 쓴다.
    import numpy as np
    import torch
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor

    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
    clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def encode(images, bs=32):
        # 그림 리스트 -> (N, 512) 단위벡터. 채점기 kit/encoders/clip_base.py 와 같은 함수다.
        out = []
        with torch.no_grad():
            for i in range(0, len(images), bs):
                b = clip_proc(images=images[i:i + bs], return_tensors="pt").to(DEV)
                f = clip_model.get_image_features(**b)
                if not torch.is_tensor(f):
                    # transformers 5.x 는 텐서 대신 결과 객체를 준다 -> 설치 셀의 버전 고정을 확인할 것
                    raise RuntimeError("transformers 버전이 5.x 다. 맨 위 설치 셀을 실행하고 "
                                       "런타임 재시작 후 다시 돌려줄 것 (4.x 로 맞춰야 한다)")
                out.append((f / f.norm(dim=-1, keepdim=True)).cpu().numpy())
        return np.concatenate(out).astype("float32")

    print("CLIP 준비 완료:", DEV)


In [ ]:
if not MOCK_CORPUS:
    # 클래스당 표본 30장을 임베딩한다 (참조 고르기엔 충분, CPU 코랩도 몇 분이면 됨).
    SAMPLE = 30
    random.seed(0)                    # 표본을 고정해야 돌릴 때마다 같은 참조가 나온다
    emb_cache = {}                    # {클래스: (파일 목록, 임베딩 행렬)}
    for style in sorted(by_style):
        files = by_style[style]
        picks = random.sample(files, min(SAMPLE, len(files)))
        imgs = [Image.open(f"{PACK}/{f}").convert("RGB") for f in picks]
        emb_cache[style] = (picks, encode(imgs))
        print(f"  {style:28s} {len(picks)}장 임베딩")
    print("완료")


In [ ]:
def pick_references(style, k=3):
    """그림체 클래스에서 '제일 그 그림체다운' k장을 골라 (파일경로, 중심과의 유사도) 리스트로 돌려준다."""
    files, E = emb_cache[style]
    center = E.mean(axis=0)
    center = center / np.linalg.norm(center)      # 중심도 단위벡터로
    sims = E @ center                             # 코사인 유사도 (전부 단위벡터라 내적이면 된다)
    order = sims.argsort()[::-1][:k]              # 유사도 큰 순 k개
    return [(files[i], float(sims[i])) for i in order]

if not MOCK_CORPUS:
    # 눈으로 한 번 확인: 세 도메인에서 하나씩 뽑아본다
    for s in ["ink_m1", "paint_Ukiyo_e", "vec_irasutoya"]:
        refs = pick_references(s)
        print(s, "->", [(f.split('/')[-1][:24], round(v, 3)) for f, v in refs])


## 2. 파이프라인에 꽂기

1편과 같은 그래프인데 **그림체 주입 노드 속만** 바뀐다: 문구 사전 대신 위의 `pick_references()`.
state 에 `refs`(찾아온 참조 이미지들) 칸이 하나 늘어난 게 전부다.

여기가 이 구조의 좋은 점인 것 같다 — **노드 속만 갈아끼우고 뼈대는 안 건드린다.**
나중에 임베딩 쪽에서 더 좋은 encode() 가 나오면 이 노드 속만 또 갈아끼우면 된다.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ComicState(TypedDict):
    story: str        # 입력: 한 줄 이야기
    style_name: str   # 쓰기로 한 그림체 클래스 (예: ink_m1, vec_irasutoya)
    cuts: list        # (1) 4컷 대본
    refs: list        # (2) ★새 칸: 검색된 참조 이미지 [(파일경로, 유사도), ...]
    prompts: list     # (2) 참조가 실린 이미지 프롬프트 4개
    images: list      # (3) 생성 이미지 (아직 자리표시)


In [ ]:
import json as _json

def mock_cuts(story):
    # 1편과 같은 흉내 대본 (키 없을 때)
    return [
        {"컷": 1, "장면": f"{story} - 상황이 시작되는 장면", "대사": "어? 이게 뭐지"},
        {"컷": 2, "장면": f"{story} - 일이 커지는 장면", "대사": "잠깐, 이러면 안 되는데"},
        {"컷": 3, "장면": f"{story} - 제일 곤란해지는 장면", "대사": "큰일 났다!"},
        {"컷": 4, "장면": f"{story} - 마무리되는 장면", "대사": "휴, 어떻게든 됐다"},
    ]

def write_cuts(state: ComicState):
    # 1편과 같은 대본 노드: 키가 있으면 LLM, 실패하거나 없으면 목으로 물러난다
    if MOCK:
        return {"cuts": mock_cuts(state["story"])}
    try:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, api_key=OPENAI_API_KEY)
        지시 = (
            "다음 이야기를 4컷 만화 대본으로 만들어 주세요. "
            '반드시 JSON 배열만 출력: [{"컷": 1, "장면": "...", "대사": "..."}, ...] 4개. '
            "장면은 그림으로 그릴 수 있게 구체적으로, 대사는 한 줄로 짧게.\n이야기: " + state["story"]
        )
        답 = llm.invoke(지시).content
        답 = 답.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
        cuts = _json.loads(답)
        assert len(cuts) == 4
        return {"cuts": cuts}
    except Exception as e:
        print("LLM 호출/파싱 실패 -> 목 대본으로 대체합니다:", e)
        return {"cuts": mock_cuts(state["story"])}


In [ ]:
# ---------- ★이번 편의 주인공: 그림체 주입 = 진짜 검색 ----------
# 1편에서 문구 사전이던 자리에 pick_references() 가 들어간다.

STYLE_WORDS = {   # 목 모드용 (1편 그대로) + 프롬프트에 얹을 말
    "ink_m1": "흑백 잉크 선화 그림체",
    "paint_Ukiyo_e": "우키요에 판화 그림체",
    "vec_irasutoya": "둥글둥글한 벡터 일러스트 그림체",
}

def inject_style(state: ComicState):
    말 = STYLE_WORDS.get(state["style_name"], state["style_name"])
    if MOCK_CORPUS:
        # 코퍼스가 없으면 1편 방식(문구만) 그대로
        return {"refs": [],
                "prompts": [f"{말}. 장면: {c['장면']}" for c in state["cuts"]]}
    refs = pick_references(state["style_name"], k=3)          # ★진짜 검색
    ref_names = ", ".join(f.split('/')[-1] for f, _ in refs)
    # 실전 생성 API 에는 참조 '이미지 자체'를 실어 보낸다. 자리표시 단계라 프롬프트에는 이름만 남긴다.
    prompts = [f"{말} (참조: {ref_names}). 장면: {c['장면']}" for c in state["cuts"]]
    return {"refs": refs, "prompts": prompts}


In [ ]:
from PIL import Image, ImageDraw, ImageFont

# 자리표시 이미지 (1편과 같음 - 생성 모델은 비용 합의 후에 붙인다)
try:
    한글폰트 = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 14)
except OSError:
    한글폰트 = ImageFont.load_default()
    print("주의: 한글 폰트를 못 찾았습니다. 설치 셀을 확인하세요.")

def generate_images(state: ComicState):
    imgs = []
    for i, p in enumerate(state["prompts"], start=1):
        img = Image.new("RGB", (256, 256), (230, 230, 230))
        d = ImageDraw.Draw(img)
        d.rectangle([4, 4, 251, 251], outline=(120, 120, 120))
        d.text((12, 12), f"cut {i}", font=한글폰트, fill=(0, 0, 0))
        for j in range(8):
            줄 = p[j * 14:(j + 1) * 14]
            if 줄:
                d.text((12, 44 + j * 20), 줄, font=한글폰트, fill=(60, 60, 60))
        imgs.append(img)
    return {"images": imgs}


In [ ]:
def check_retrieval(state: ComicState):
    # 이번 편의 '채점'은 생성물이 아니라 검색 품질을 본다:
    # 찾아온 참조가 정말 요청한 클래스인지 + 중심과 얼마나 가까운지.
    # (생성물 채점은 생성이 진짜가 되는 날 kit/score.py 잣대로 붙인다)
    if MOCK_CORPUS:
        print("목 모드 - 검색 품질 확인 생략")
        return {}
    ok = all(f"images/{state['style_name']}/" in f for f, _ in state["refs"])
    sims = [round(v, 3) for _, v in state["refs"]]
    print(f"참조 {len(state['refs'])}장 | 요청 클래스와 일치: {ok} | 중심 유사도: {sims}")
    return {}


In [ ]:
# 그래프 조립 - 1편과 같은 문법. 이번엔 직선(루프는 생성이 진짜가 되면 부활).
builder = StateGraph(ComicState)
builder.add_node("대본작성", write_cuts)
builder.add_node("그림체검색", inject_style)
builder.add_node("이미지생성", generate_images)
builder.add_node("검색품질확인", check_retrieval)
builder.add_edge(START, "대본작성")
builder.add_edge("대본작성", "그림체검색")
builder.add_edge("그림체검색", "이미지생성")
builder.add_edge("이미지생성", "검색품질확인")
builder.add_edge("검색품질확인", END)
app = builder.compile()

이야기 = "지각한 학생이 교문에서 선생님과 딱 마주쳤다"
final = app.invoke({
    "story": 이야기,
    "style_name": "ink_m1",   # paint_Ukiyo_e, vec_irasutoya 로도 바꿔 돌려볼 것
    "cuts": [], "refs": [], "prompts": [], "images": [],
})
for c in final["cuts"]:
    print(f"  {c['컷']}컷 | {c['장면']} | {c['대사']}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# matplotlib 도 한글 폰트를 알려줘야 제목이 안 깨진다 (PIL 과 별개다)
try:
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

# 윗줄 = 검색으로 찾아온 참조 (컷 공통인 그림체 앵커라 한 번만 보여준다)
# 아랫줄 = 4컷 자리표시 (생성 모델이 붙으면 여기가 진짜 그림이 된다)
n_ref = len(final["refs"])
if n_ref:
    fig, axes = plt.subplots(1, n_ref, figsize=(2.6 * n_ref, 2.8), squeeze=False)
    for j, (f, sim) in enumerate(final["refs"]):
        axes[0][j].imshow(Image.open(f"{PACK}/{f}"))
        axes[0][j].axis("off")
        axes[0][j].set_title(f"참조 {j+1} (유사도 {sim:.2f})", fontsize=9)
    fig.suptitle(f"코퍼스에서 검색해 온 참조 - 그림체 {final['style_name']}", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("(목 모드라 참조가 없습니다 - 코퍼스를 연결하면 진짜 그림이 뜹니다)")

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.8), squeeze=False)
for j, (img, c) in enumerate(zip(final["images"], final["cuts"])):
    axes[0][j].imshow(img)
    axes[0][j].axis("off")
    axes[0][j].set_title(f"{c['컷']}컷 (생성될 자리)", fontsize=9)
plt.tight_layout()
plt.show()


## 3. 정리 — 여기까지 하고 안 것

1. **검색이 진짜가 됐다**: 한 줄 이야기 -> 대본 -> 코퍼스에서 참조 이미지 검색 -> (생성 자리) 까지
   실물 그림이 흐른다. 위 그리드가 그 증거
2. **뼈대는 안 바꿨다**: 1편 그래프에서 노드 속만 갈아끼웠다. 더 좋은 encode() 가 나오면
   `pick_references()` 안의 임베딩만 또 갈아끼우면 된다 (임베딩 공부와 만나는 지점)
3. **다음 숙제**: 생성 모델 연결(비용 합의 필요) -> 생성물 채점(kit/score.py 잣대) -> 재시도 루프 부활 ->
   장면 내용까지 반영한 참조 고르기(지금은 그림체만 보고 고른다)

정직하게 적어둘 한계 하나: 지금 검색은 **클래스 라벨을 알고 그 안에서 대표를 고르는 것**이지,
전체 코퍼스를 열어놓고 찾는 검색이 아니다. "라벨 없이도 그림체가 비슷한 걸 찾아오는가"는
임베딩이 그림체를 제대로 잡아야 되는 일이라, 임베딩 표현 공부와 만나는 지점이다.

`style_name` 을 `paint_Ukiyo_e`, `vec_irasutoya` 로 바꾸면 전혀 다른 참조가 걸려 나온다.
20개 클래스 명단은 위 meta 셀 출력에 있다.
